# ShopDesk, Section 4 Lab 1: Prerequisite Gating and Structured Escalation

A beginner-friendly notebook that moves a ShopDesk guarantee out of the prompt and into
**code**: a `PreToolUse` hook that blocks a refund until the customer is verified. We also
design a **structured escalation** output and compare **prompt-based** with **programmatic**
enforcement. Built on the **Claude Agent SDK**, running **Sonnet** (`claude-sonnet-4-6`)
through your **Anthropic API key**.

## The real-world scenario

"Always verify the customer before refunding" is a fine sentence to put in a prompt, and a
model will usually follow it. Usually is not good enough when real money is involved. The
same rule written as a **hook** cannot be talked out of: the refund tool simply does not run
until its prerequisite is met.

The question this lab answers: **how do you make a workflow prerequisite a guarantee rather
than a suggestion, and how do you hand off a case in a consistent, structured shape?**

## Objectives

- Enforce a **tool execution sequence** with a `PreToolUse` gate: no `refund_order` until
  `get_customer` has verified the order.
- Design a **structured escalation** output (customer details, root cause, recommended
  actions) with a JSON schema.
- **Compare** prompt-based enforcement (best effort) with programmatic enforcement
  (deterministic).

## What you'll observe

- In pure Python, an ungated refund runs even when unverified; a gated one is blocked until
  the prerequisite is met.
- Live, a cold refund is denied by the hook, then the same refund succeeds after a verify.
- The escalation builder produces a validated, consistent packet every time.

## How to run

Run top to bottom. The enforcement contrast and the escalation cells are pure Python and run
anywhere. The hook-guarded cells call Claude, so paste a real key into **Setup 2/3** and
re-run from the top; otherwise they skip. **Node.js 18+** must be installed for the Agent
SDK.

## 0. Setup

**This cell:** installs the packages. The **Agent SDK** provides the tools and hooks;
the base SDK and dotenv handle the key. The Agent SDK also needs Node.js 18+, which cannot be
pip-installed.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` so the async
guarded runs can be called like ordinary functions.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # read the API key from the environment
import sys                                      # detect Windows (it needs a special event loop)
import json                                     # build tool payloads and escalation packets
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the agent will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds the shared **ShopDesk world**: orders and their customers. The
verify tool reads `CUSTOMERS`, and the gate uses a `VERIFIED` set as its rail.

In [ ]:
# ===== SETUP 3/3 - the shared ShopDesk data and the rail =====
ORDERS = {"A1": {"status": 2}, "A2": {"status": 3}}   # a tiny order book
CUSTOMERS = {"A1": {"name": "Ravi"}, "A2": {"name": "Meera"}}   # who owns each order
STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}  # status code -> word

def refund_order(order_id):                      # the plain refund action (used offline below)
    return "refunded " + order_id               #   in this lab the gate, not the body, is the point

VERIFIED = set()                                 # orders whose customer has been verified (the rail)
print("orders:", list(ORDERS), "| verified starts empty")

**This cell:** the **narrator** `stream_run()`, which runs one `query()` and prints each
tool call and the final text. The hooks themselves also print when they fire, so you can see
the gate act.

In [ ]:
# ===== stream one guarded run and narrate it =====
from claude_agent_sdk import (                     # the Agent SDK pieces we use:
    query, ClaudeAgentOptions,                     #   run + options
    tool, create_sdk_mcp_server, HookMatcher,      #   define tools, bundle them, wire hooks
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,   # message + block types
)

async def stream_run(options, prompt):             # run query() and print what happens
    print("USER:", prompt)                         #   echo the request
    answer = ""                                    #   keep the final text
    async for message in query(prompt=prompt, options=options):   # stream every message
        if isinstance(message, AssistantMessage):  #     the model spoke
            for block in message.content:          #       walk its blocks
                if isinstance(block, ToolUseBlock):#       a tool call...
                    print("  -> tool:", block.name.split("__")[-1], block.input)
                elif isinstance(block, TextBlock): #       ...or text
                    answer = block.text            #         remember the latest text
        elif isinstance(message, ResultMessage):    #     the run finished
            pass
    print("ANSWER:", answer)                        #   the final answer
    return answer

### Prompt-only versus programmatic enforcement

A rule in a **prompt** is advisory: the model reads it and usually complies, but nothing stops
it from skipping the step. A rule in **code** (a hook) is a control point in the execution
lifecycle: it runs before the tool and can **deny** it outright, so the prerequisite holds
every time regardless of wording. Money-touching steps belong in code.

---

### 🎯 Lab objective - make the prerequisite a guarantee

**What you build:** a `PreToolUse` gate that blocks `refund_order` until the order is verified,
a `PostToolUse` hook that records the verification, and a structured escalation packet.

**Why it helps you build real solutions:** a prompt can be argued out of a rule; a hook cannot.
Moving guarantees into code is what makes an agent safe with money and consistent under
pressure.

**How you'll see it:** the first refund is denied, the verified one succeeds, and the escalation
packet validates every time.

**This cell:** the enforcement contrast in pure Python, part one. `unguarded_refund`
models the prompt-only world: there is no code check, so the refund runs even when the order is
not verified. `guarded_refund` adds the prerequisite check in code. Both run offline.

In [ ]:
# ===== prompt-only vs programmatic, in plain Python =====
def unguarded_refund(order_id, verified):          # prompt-only world: the rule lives only in text
    return refund_order(order_id)                  #   nothing in CODE stops an unverified refund

def guarded_refund(order_id, verified):            # programmatic world: the rule lives in code
    if order_id not in verified:                   #   the prerequisite, checked deterministically
        return "BLOCKED: verify the customer first"#     denied, every time
    return refund_order(order_id)                  #   only runs once verified

**This cell:** part two runs both on an **unverified** order, then on a verified one. The
ungated version refunds regardless (the risk a prompt-only rule leaves open); the gated version
blocks until the prerequisite is met. This is deterministic and needs no key.

In [ ]:
# ===== show the difference =====
verified = set()                                   # nobody verified yet
print("unguarded, unverified:", unguarded_refund("A1", verified))   # refunds anyway (bad)
print("guarded,   unverified:", guarded_refund("A1", verified))     # blocked (good)
verified = {"A1"}                                  # now A1 is verified
print("guarded,   verified:  ", guarded_refund("A1", verified))     # allowed

**This cell:** the real **Agent SDK tools**. `get_customer` verifies (looks up the owner);
`refund_order` is the guarded action. Defining them with `@tool` is what lets the hooks below sit
in front of them.

In [ ]:
# ===== the SDK tools =====
@tool("get_customer", "Look up the customer for an order.", {"order_id": str})
async def sdk_get_customer(args):                  # the verifying tool
    oid = args["order_id"]                         #   which order
    c = CUSTOMERS.get(oid, {"name": "unknown"})    #   look up (or a miss)
    return {"content": [{"type": "text", "text": json.dumps(c)}]}   # MCP result shape

@tool("refund_order", "Refund an order.", {"order_id": str})
async def sdk_refund_order(args):                  # the guarded tool
    return {"content": [{"type": "text", "text": "refunded " + args["order_id"]}]}

**This cell:** the **`PreToolUse` gate**. It runs *before* any tool. If the tool is
`refund_order` and the order is not in `VERIFIED`, it returns a `deny` decision, so the refund
never executes. Returning an empty dict allows the call.

In [ ]:
# ===== the gate: deny a refund before verification =====
async def gate_refund(input_data, tool_use_id, context):   # runs BEFORE a tool
    name = input_data["tool_name"].split("__")[-1]         #   the bare tool name
    oid = input_data.get("tool_input", {}).get("order_id") #   which order
    if name == "refund_order" and oid not in VERIFIED:     #   refund without a verify?
        print("  [gate] refund of", oid, "blocked: not verified")   # show the gate firing
        return {"hookSpecificOutput": {                    #   -> DENY the tool
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": "Verify the customer first."}}
    return {}                                              #   otherwise allow

**This cell:** the **`PostToolUse` recorder**. It runs *after* a tool. When a
`get_customer` verify succeeds, it adds that order to `VERIFIED`, which is what later unlocks the
refund. The rule now lives entirely in code.

In [ ]:
# ===== record a verification so the gate can later allow the refund =====
async def record_verify(input_data, tool_use_id, context):   # runs AFTER a tool
    name = input_data["tool_name"].split("__")[-1]           #   the bare tool name
    oid = input_data.get("tool_input", {}).get("order_id")   #   which order
    if name == "get_customer" and oid:                       #   a verify just happened
        VERIFIED.add(oid)                                    #     record it -> unlocks refund
        print("  [hook] recorded verify for", oid)           #   show it firing
    return {}                                                #   observe only, never blocks

**This cell:** bundles the tools and wires both hooks into the options with `HookMatcher`.
A matcher with no pattern fires for every tool; our hooks check the tool name themselves. This
options object is the enforced workflow.

In [ ]:
# ===== the guarded options =====
shop = create_sdk_mcp_server(name="shop", version="1.0.0",   # bundle the two tools
                             tools=[sdk_get_customer, sdk_refund_order])
GUARDED = ClaudeAgentOptions(                                 # options with the hooks wired in
    model=MODEL, mcp_servers={"shop": shop},
    allowed_tools=["mcp__shop__get_customer", "mcp__shop__refund_order"],
    hooks={"PreToolUse":  [HookMatcher(hooks=[gate_refund])],     # enforce before
           "PostToolUse": [HookMatcher(hooks=[record_verify])]})  # record after
print("guarded workflow ready")

**This cell:** runs a **cold refund** (no verify first). Watch the gate print its block and
the model report that it cannot refund yet. The prerequisite held without any prompt saying so.

In [ ]:
# ===== run: refund before verify -> denied =====
if RUN_LIVE:                                      # needs a real key (and Node.js 18+)
    VERIFIED.clear()                              #   start unverified
    run_async(lambda: stream_run(GUARDED, "Refund order A1."))   # -> gate denies it
else:
    print("[skipped] expected: the [gate] blocks the refund because A1 is not verified.")

**This cell:** runs the **verify-then-refund** flow. The `get_customer` call verifies A1
(recorded by the PostToolUse hook), which unlocks the refund on the next step. Same gate,
opposite outcome, decided entirely by the order of tool calls.

In [ ]:
# ===== run: verify first, then refund -> allowed =====
if RUN_LIVE:                                      # needs a real key
    VERIFIED.clear()                              #   start unverified again
    run_async(lambda: stream_run(GUARDED, "Look up the customer for A1, then refund A1."))
else:
    print("[skipped] expected: verify records A1, then the refund is allowed through.")

**This cell:** the **prompt-only** comparison. Same tools, but the rule lives only in the
system prompt and there is **no gate**. The model will usually comply, yet nothing in code stops
a cold refund, which is exactly the reliability gap the hook closes.

In [ ]:
# ===== prompt-only enforcement: advisory, not guaranteed =====
PROMPT_ONLY = ClaudeAgentOptions(                  # NO hooks; the rule is only in the prompt
    model=MODEL, mcp_servers={"shop": shop},
    allowed_tools=["mcp__shop__get_customer", "mcp__shop__refund_order"],
    system_prompt="Always verify the customer with get_customer before any refund_order.")
if RUN_LIVE:                                       # needs a real key
    VERIFIED.clear()                               #   the prompt-only run ignores VERIFIED entirely
    run_async(lambda: stream_run(PROMPT_ONLY, "Refund order A1."))   # may or may not comply
else:
    print("[skipped] expected: usually complies, but NOTHING in code prevents a cold refund.")

**This cell:** the **structured escalation** schema. When a case needs a human, a
consistent shape (customer details, root cause, recommended actions, priority) makes the hand-off
reliable. This JSON schema is the contract the builder fills.

In [ ]:
# ===== the structured escalation contract =====
ESCALATION_SCHEMA = {                              # the shape every escalation must take
    "type": "object",
    "properties": {
        "customer": {"type": "object",             #   who this is about
                     "properties": {"name": {"type": "string"}, "order_id": {"type": "string"}},
                     "required": ["name", "order_id"]},
        "root_cause": {"type": "string"},          #   why it is being escalated
        "recommended_actions": {"type": "array", "items": {"type": "string"}},   # what to do
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},        # how urgent
    },
    "required": ["customer", "root_cause", "recommended_actions", "priority"],
}
print("escalation fields:", list(ESCALATION_SCHEMA["properties"]))

**This cell:** a **builder** that fills the contract and a tiny **validator** that checks the
required fields are present. Building the packet in code (not free prose) is what keeps every
hand-off consistent. It runs offline.

In [ ]:
# ===== build and validate a structured escalation =====
def build_escalation(order_id, root_cause, actions, priority):   # fill the contract
    return {
        "customer": {"name": CUSTOMERS.get(order_id, {}).get("name", "unknown"), "order_id": order_id},
        "root_cause": root_cause,                  #   the reason
        "recommended_actions": actions,            #   the next steps
        "priority": priority,                      #   the urgency
    }

def validate_escalation(pkt):                      # check the required fields exist
    ok = all(k in pkt for k in ESCALATION_SCHEMA["required"]) and \
         all(k in pkt["customer"] for k in ["name", "order_id"])
    return ok

esc = build_escalation("A2", "Refund refused past 30-day window; customer upset.",
                       ["Offer store credit", "Have a human call within 24h"], "high")
print(json.dumps(esc, indent=2))                   # the consistent packet
print("valid:", validate_escalation(esc))          # the validator agrees

| anti-pattern | what to do instead |
|---|---|
| trust a prompt to enforce a money rule | gate the tool in code with a `PreToolUse` deny |
| track prerequisites in the prompt | record them in code (a `PostToolUse` hook + a set) |
| hand off cases as free prose | fill a structured schema so every hand-off is consistent |
| return a raw "deny" with no reason | include a clear `permissionDecisionReason` the model can act on |

**Lesson:** the prompt asks; your **code decides**. A `PreToolUse` gate turns "verify before
you refund" from a suggestion into a guarantee, a `PostToolUse` hook records the prerequisite, and
a structured escalation schema keeps every hand-off consistent. Prompt-only enforcement is best
effort; programmatic enforcement is deterministic.

---

## Recap - guarantees in code

| Piece | In this lab | Course topic |
|---|---|---|
| PreToolUse gate | deny `refund_order` until verified | programmatic prerequisite enforcement |
| PostToolUse recorder | mark an order verified after `get_customer` | control points in the lifecycle |
| Enforcement contrast | ungated runs vs gated blocks | prompt-based vs programmatic reliability |
| Escalation schema | customer, root cause, actions, priority | structured handoff patterns |

One principle to carry forward: **put money-touching rules in code, not the prompt, and hand off
in a fixed shape.** To run live, paste a real key into **Setup 2/3** and re-run from the top. Then
try it: word the cold refund as an emergency and watch the gate still deny it. Next lab: PostToolUse
normalization and compliance thresholds.